# rank-world-size-args — faded example 1: Broadcast: src sends to all non-src ranks

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank-world-size-args`. Running the beacon reports progress on the `Distributed: rank/world_size args` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank/world_size args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank-world-size-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank-world-size-args"
DD_SUBTOPIC = "Distributed: rank/world_size args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In a broadcast, rank `src` sends the tensor to every other rank, and all other ranks receive from `src`. The function signature `broadcast_protocol(tensor, rank, world_size, src=0)` follows the distributed primitive convention where `rank` and `world_size` are always passed explicitly so each spawned process can determine its role.

## Faded exercise 1

### Exercise — Broadcast protocol: src sends to all non-src ranks

Complete `broadcast_protocol(tensor, rank, world_size, src=0)`. When `rank == src`, return a list of send actions to every other rank (ascending). When `rank != src`, return a single receive from `src`.

Fill in the src-rank send list.

**Fill in:** Return the list of ('send', other_rank) tuples for every other rank in ascending order, when this rank is the source.

In [ ]:
import torch as t

def broadcast_protocol(tensor, rank: int, world_size: int, src: int = 0) -> list:
    if rank == src:
        return [('send', other) for other in range(world_size) if other != src]
    return [('recv', src)]

# Quick check
for r in range(3):
    print(r, broadcast_protocol(None, r, 3))


def _test():
    # Src=0, world_size=4
    src_actions = broadcast_protocol(None, 0, 4)
    assert src_actions == [('send', 1), ('send', 2), ('send', 3)]
    # Non-src ranks
    for r in range(1, 4):
        actions = broadcast_protocol(None, r, 4)
        assert actions == [('recv', 0)], f'rank {r} should recv from 0'
    # Non-default src
    src2_actions = broadcast_protocol(None, 2, 4, src=2)
    assert ('send', 0) in src2_actions
    assert ('send', 1) in src2_actions
    assert ('send', 3) in src2_actions
    assert len(src2_actions) == 3


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def broadcast_protocol(tensor, rank: int, world_size: int, src: int = 0) -> list:
    if rank == src:
        return [('send', other) for other in range(world_size) if other != src]
    return [('recv', src)]

# Quick check
for r in range(3):
    print(r, broadcast_protocol(None, r, 3))
```
</details>